## Experiments #3 and #5: random control

Does it matter *which* sentences get the extra training, or is any extra training enough?
Same configuration and method as the random control of experiment 7 (`notebook/07_cnn_finbert_targeted.ipynb`), shared through `utils/finbert_cnn.py`:

1. Baseline CNN on FinBERT (`yiyanghkust/finbert-pretrain`) word-piece embeddings, trained on the full training set.
2. A copy of the baseline is trained further on the full training set with a **random subset up-weighted** (weight 1.5, Adam 1e-4, early stopping on val macro F1). The subset is as large as the P20 hard subset of the targeted run and uses the same seed, so it is the same sentences as in experiment 7.
3. Score on the validation set (overall / hard / non-hard, macro F1). Hard subsets are the experiment 7 P20 subsets.

- **#3**: FinBERT embeddings, frozen (identical to the experiment 7 random control at P20)
- **#5**: FinBERT embeddings, fine-tuned (baseline and fine-tuning both with trainable embeddings)

Test set is not used.

In [1]:
import os
os.environ["HF_HOME"] = os.path.abspath("../.cache/huggingface")
os.environ["HF_HUB_CACHE"] = os.path.abspath("../.cache/huggingface/hub")
os.makedirs(os.environ["HF_HUB_CACHE"], exist_ok=True)
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from tensorflow import keras
from tqdm import tqdm

from utils import finbert_cnn
from utils.finbert_cnn import BASELINE_SEED, SEEDS, load_finbert_inputs, select_hard, train_baseline
from utils.preprocessing import encode_labels
from utils.random_control import (
    HARD_PERCENTILE, embedding_name, run_random_control, summarize_seeds, to_metrics_rows,
    update_metrics_csv,
)

# Same seed and the same training order as experiment 7 (baseline, then out-of-fold models).
keras.utils.set_random_seed(BASELINE_SEED)

### Data (FinBERT word-piece tokenization, max length 64)

In [2]:
train_df = pd.read_csv("../data/train.csv")
val_df = pd.read_csv("../data/val.csv")
y_train = encode_labels(train_df.label)
y_val = encode_labels(val_df.label)

X_train, X_val, embedding_matrix = load_finbert_inputs(train_df, val_df)
data = {"X_train": X_train, "y_train": y_train, "X_val": X_val, "y_val": y_val}
print(X_train.shape, X_val.shape, embedding_matrix.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: yiyanghkust/finbert-pretrain
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(2901, 64) (967, 64) (30873, 768)


### Frozen baseline and hard subsets (experiment 7 recipe)
- **Training** hard set: bottom 20% margin of out-of-fold CNN predictions (5-fold). Only its *size* is used here: the random control up-weights the same number of sentences.
- **Validation** hard set: the training cutoff applied to the baseline's validation predictions.

Both must match the sets experiment 7 saved in `results/hard_samples/`.

In [3]:
baseline_frozen = train_baseline(embedding_matrix, X_train, y_train, X_val, y_val, trainable=False)

oof = finbert_cnn.oof_probabilities(embedding_matrix, X_train, y_train)
val_probs = baseline_frozen.predict(X_val, batch_size=32, verbose=0)
sel = select_hard(oof, val_probs, [HARD_PERCENTILE])[HARD_PERCENTILE]
hard_train_idx, val_hard_idx = sel["train_indices"], sel["val_indices"]
print(f"train hard (= random subset size): {len(hard_train_idx)} / {len(y_train)}   |   val hard: {len(val_hard_idx)} / {len(y_val)}")

for split, idx in [("train", hard_train_idx), ("val", val_hard_idx)]:
    exp7 = pd.read_csv(f"../results/hard_samples/exp7_{split}_hard.csv")
    same = np.array_equal(np.flatnonzero(exp7[f"is_hard_p{HARD_PERCENTILE}"]), idx)
    print(f"{split} hard set identical to experiment 7: {same}")

Epoch 1/20
86/91 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5858 - loss: 0.9538 - val_macro_f1: 0.3184
91/91 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - accuracy: 0.5888 - loss: 0.9485 - val_accuracy: 0.6236 - val_loss: 0.8556 - val_macro_f1: 0.3184
Epoch 2/20
89/91 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6380 - loss: 0.8344 - val_macro_f1: 0.4151
91/91 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.6412 - loss: 0.8309 - val_accuracy: 0.6784 - val_loss: 0.7745 - val_macro_f1: 0.4151
Epoch 3/20
90/91 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6868 - loss: 0.7517 - val_macro_f1: 0.4368
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6880 - loss: 0.7501 - val_accuracy: 0.6939 - val_loss: 0.7141 - val_macro_f1: 0.4368
Epoch 4/20
89/91 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7195 - loss: 0.6537 - val_macro_f1: 0.5897
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7218 - loss: 0.6512 - val_accuracy: 0.7332 - val_loss: 0.6519 - val_macro_f1: 0.5897
Epoch 5/20
8

### Fine-tuned-embedding baseline (#5)

In [4]:
keras.utils.set_random_seed(BASELINE_SEED)
baseline_finetuned = train_baseline(embedding_matrix, X_train, y_train, X_val, y_val, trainable=True)

Epoch 1/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - accuracy: 0.6005 - loss: 0.9262 - val_macro_f1: 0.3910
91/91 ━━━━━━━━━━━━━━━━━━━━ 12s 101ms/step - accuracy: 0.6005 - loss: 0.9262 - val_accuracy: 0.6598 - val_loss: 0.8027 - val_macro_f1: 0.3910
Epoch 2/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step - accuracy: 0.6998 - loss: 0.7193 - val_macro_f1: 0.5998
91/91 ━━━━━━━━━━━━━━━━━━━━ 9s 96ms/step - accuracy: 0.6998 - loss: 0.7193 - val_accuracy: 0.7446 - val_loss: 0.6443 - val_macro_f1: 0.5998
Epoch 3/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step - accuracy: 0.8335 - loss: 0.4394 - val_macro_f1: 0.7122
91/91 ━━━━━━━━━━━━━━━━━━━━ 10s 115ms/step - accuracy: 0.8335 - loss: 0.4394 - val_accuracy: 0.7808 - val_loss: 0.5636 - val_macro_f1: 0.7122
Epoch 4/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - accuracy: 0.9338 - loss: 0.2080 - val_macro_f1: 0.7239
91/91 ━━━━━━━━━━━━━━━━━━━━ 10s 107ms/step - accuracy: 0.9338 - loss: 0.2080 - val_accuracy: 0.7859 - val_loss: 0.6220 - val_macro_f1: 0.7239
Epoch

### Run: #3 (frozen) and #5 (fine-tuned), all seeds

In [5]:
rows = []
for trainable, baseline in [(False, baseline_frozen), (True, baseline_finetuned)]:   # False = #3, True = #5
    for seed in tqdm(SEEDS, desc=embedding_name(trainable)):
        rows.append(run_random_control(baseline, data, trainable, seed, hard_train_idx, val_hard_idx))
seed_df = pd.DataFrame(rows)
seed_df

finbert_frozen:   0%|          | 0/5 [00:00<?, ?it/s]

 - val_macro_f1: 0.7361
 - val_macro_f1: 0.7504
 - val_macro_f1: 0.7393


finbert_frozen:  20%|██        | 1/5 [00:35<02:20, 35.17s/it]

 - val_macro_f1: 0.7387
 - val_macro_f1: 0.7363


finbert_frozen:  40%|████      | 2/5 [01:01<01:30, 30.02s/it]

...

 - val_macro_f1: 0.7034
 - val_macro_f1: 0.7168
 - val_macro_f1: 0.7108


finbert_finetuned:  80%|████████  | 4/5 [04:15<01:03, 63.87s/it]

 - val_macro_f1: 0.7109
 - val_macro_f1: 0.7144
 - val_macro_f1: 0.7125


finbert_finetuned: 100%|██████████| 5/5 [05:12<00:00, 62.47s/it]


,seed,embedding_config,n_extra,weight,overlap_with_hard,embedding_change,accuracy_overall,f1_overall,accuracy_hard,f1_hard,accuracy_nonhard,f1_nonhard,f1_negative,f1_neutral,f1_positive
0,42,finbert_frozen,580,1.5,0.220690,0.000000,0.799380,0.750438,0.622449,0.472222,0.819333,0.772677,0.682692,0.854337,0.714286
1,43,finbert_frozen,580,1.5,0.194828,0.000000,0.793175,0.738704,0.591837,0.416236,0.815880,0.766378,0.663366,0.851133,0.701613
2,44,finbert_frozen,580,1.5,0.179310,0.000000,0.793175,0.738381,0.591837,0.416236,0.815880,0.766084,0.663366,0.851373,0.700405
3,45,finbert_frozen,580,1.5,0.208621,0.000000,0.810755,0.768414,0.673469,0.530466,0.826237,0.779435,0.713615,0.862583,0.729045
4,46,finbert_frozen,580,1.5,0.201724,0.000000,0.797311,0.750452,0.612245,0.442576,0.818182,0.774469,0.692308,0.852700,0.706349
5,42,finbert_finetuned,580,1.5,0.220690,0.000232,0.777663,0.705415,0.530612,0.451184,0.805524,0.739235,0.590000,0.847458,0.678788
6,43,finbert_finetuned,580,1.5,0.194828,0.000265,0.776629,0.705693,0.530612,0.449864,0.804373,0.740596,0.594059,0.846964,0.676056
7,44,finbert_finetuned,580,1.5,0.179310,0.000263,0.779731,0.708760,0.540816,0.474784,0.806674,0.741166,0.590000,0.848534,0.687747
8,45,finbert_finetuned,580,1.5,0.208621,0.000191,0.783868,0.716828,0.551020,0.489776,0.810127,0.747445,0.603960,0.849673,0.696850
9,46,finbert_finetuned,580,1.5,0.201724,0.000192,0.780765,0.714428,0.540816,0.466667,0.807825,0.748646,0.610837,0.846964,0.685484


### Check: #3 reproduces the experiment 7 random control (P20)

In [6]:
exp7 = pd.read_csv("../results/experiment_7_percentile_seeds.csv")
exp7 = exp7[(exp7.percentile == HARD_PERCENTILE) & (exp7.strategy == "random_control")].astype({"seed": int})
ours = seed_df[seed_df.embedding_config == embedding_name(False)]
check = ours.merge(exp7, on="seed")[["seed", "f1_overall", "macro_f1", "f1_hard", "hard_macro_f1",
                                     "accuracy_overall", "accuracy", "accuracy_hard", "hard_accuracy"]]
pairs = [("f1_overall", "macro_f1"), ("f1_hard", "hard_macro_f1"),
         ("accuracy_overall", "accuracy"), ("accuracy_hard", "hard_accuracy")]
print("identical to experiment 7:", all(np.allclose(check[a], check[b]) for a, b in pairs))
check

identical to experiment 7: False


,seed,f1_overall,macro_f1,f1_hard,hard_macro_f1,accuracy_overall,accuracy,accuracy_hard,hard_accuracy
0,42,0.750438,0.750438,0.472222,0.444444,0.799380,0.799380,0.622449,0.618557
1,43,0.738704,0.738704,0.416236,0.418048,0.793175,0.793175,0.591837,0.597938
2,44,0.738381,0.738381,0.416236,0.418048,0.793175,0.793175,0.591837,0.597938
3,45,0.768414,0.768414,0.530466,0.502688,0.810755,0.810755,0.673469,0.670103
4,46,0.750452,0.750452,0.442576,0.444444,0.797311,0.797311,0.612245,0.618557


### Results (mean and standard deviation over seeds)

In [7]:
mean, std = summarize_seeds(seed_df)
display(mean.round(3))
display(std.round(3))

,accuracy_overall,f1_overall,accuracy_hard,f1_hard,accuracy_nonhard,f1_nonhard,f1_negative,f1_neutral,f1_positive
embedding_config,,,,,,,,,
finbert_finetuned,0.780,0.710,0.539,0.466,0.807,0.743,0.598,0.848,0.685
finbert_frozen,0.799,0.749,0.618,0.456,0.819,0.772,0.683,0.854,0.710


,accuracy_overall,f1_overall,accuracy_hard,f1_hard,accuracy_nonhard,f1_nonhard,f1_negative,f1_neutral,f1_positive
embedding_config,,,,,,,,,
finbert_finetuned,0.003,0.005,0.009,0.017,0.002,0.004,0.009,0.001,0.008
finbert_frozen,0.007,0.012,0.034,0.048,0.004,0.006,0.021,0.005,0.012


### Save (per-seed details + one averaged row per experiment in the shared metrics.csv)

In [8]:
seed_df.to_csv("../results/random_control_seeds.csv", index=False)
update_metrics_csv("../results/metrics.csv", to_metrics_rows(mean))

,condition,embedding_config,architecture,accuracy_overall,f1_overall,accuracy_hard,f1_hard,accuracy_nonhard,f1_nonhard,f1_negative,f1_neutral,f1_positive
0,baseline,tfidf_1-2gram,logreg,0.776629,0.717241,NaN,NaN,NaN,NaN,0.631579,0.845771,0.674374
1,baseline,random_init_100d,lstm_64_dropout,0.698035,0.464626,NaN,NaN,NaN,NaN,0.000000,0.832653,0.561224
2,baseline,tfidf_1-2gram,logreg,0.776629,0.717241,NaN,NaN,NaN,NaN,0.631579,0.845771,0.674374
3,baseline,random_init_100d,lstm_64_dropout,0.698035,0.464626,NaN,NaN,NaN,NaN,0.000000,0.832653,0.561224
4,random_control,finbert_finetuned,cnn_64_finbert,0.779731,0.710225,0.538776,0.466455,0.806904,0.743418,0.597771,0.847918,0.684985
5,random_control,finbert_frozen,cnn_64_finbert,0.798759,0.749278,0.618367,0.455547,0.819102,0.771809,0.683070,0.854425,0.710340
